# Mapa 5 - Comparacion territorial de Barrio Obrero (Cali)

Mapa interactivo que compara **tres delimitaciones espaciales** de la zona
de estudio Barrio Obrero usando el **IPM** (Indice de Pobreza
Multidimensional) a nivel de manzana:

1. **Poligono V1** (16 manzanas, limite preliminar).
2. **Poligono oficial** (45 manzanas, limite ampliado).
3. **Poligono aledano** (seleccion dinamica): el usuario dibuja un circulo
   o poligono libre con la herramienta de dibujo; las manzanas cuyo
   centroide quede contenido se agregan y promedian en vivo (turf.js).

**Nota sobre el ITT:** las variables del ITT (seguridad, movilidad, etc.)
no existen para Barrio Obrero (Comuna 9). El dataset ITT real solo cubre
Ciudad Paraiso (Comuna 3). Por eso el bloque ITT del panel se muestra
rotulado **"Sin datos ITT para Comuna 9 (pendiente de DATIC)"** y la
comparacion efectiva es sobre el IPM y sus privaciones.

- Paleta accesible Okabe-Ito: V1 azul `#0072B2`, oficial verde `#009E73`,
  aledano naranja `#E69F00`.
- Unidad del IPM: % de personas. Vacios = 0. Fuente del dato: Censo 2018
  (DANE, CNPV). Salida: `mapa_html/Barrio_obrero/mapa_comparacion_barrio_obrero.html`.


## 0. Setup Colab (opcional)

In [ ]:
# === SETUP COLAB (se ignora en local) ===
def _en_colab():
    try:
        import google.colab
        return True
    except Exception:
        return False
if _en_colab():
    import os,subprocess,sys
    REPO='https://github.com/j0rg3c45/mapa_Analitico_ciudad_Paraiso.git'
    DEST='/content/mapa_Analitico_ciudad_Paraiso'
    subprocess.run([sys.executable,'-m','pip','install','-q','geopandas','folium','mapclassify','pyarrow','openpyxl'],check=False)
    if not os.path.isdir(DEST): subprocess.run(['git','clone','--depth','1',REPO,DEST],check=False)
    if os.path.isdir(os.path.join(DEST,'notebooks')): os.chdir(os.path.join(DEST,'notebooks'))
    print('Colab listo:',os.getcwd())
else:
    print('Entorno local (uv). Setup Colab omitido.')


## 1. Entorno y paleta Okabe-Ito

In [ ]:
import os
import json
import numpy as np
import pandas as pd
import geopandas as gpd
# Paleta Okabe-Ito (apta daltonismo) para las 3 series del comparativo.
OKI_AZUL="#0072B2"    # Serie 1: Poligono V1
OKI_VERDE="#009E73"   # Serie 2: Poligono oficial (45 manzanas)
OKI_NARANJA="#E69F00" # Serie 3: Poligono aledano (seleccion dinamica)
OKI_BERMELL="#D55E00"
OKI_GRIS="#333333"; BG="#F4F6F9"; CRS_WEB=4326
print("Entorno listo.")

## 2. Rutas, poligonos y salida

In [ ]:
def _ruta_bo(*partes):
    for base in [os.path.join("..","data","Barrio_Obrero_geojson"),
                 os.path.join("data","Barrio_Obrero_geojson"),
                 os.path.join("/content","data","Barrio_Obrero_geojson")]:
        p=os.path.join(base,*partes)
        if os.path.isfile(p): return os.path.abspath(p)
    return None
_p_v1  =_ruta_bo("poligono_Barrio_Obrero_V1.geojson")
_p_of  =_ruta_bo("Geojson_Barrio_Obrero.geojson")
_p_ipm =_ruta_bo("IPM_Barrio_Obrero_completo.geojson")
for _nm,_pp in [("V1",_p_v1),("oficial",_p_of),("IPM",_p_ipm)]:
    if _pp is None: raise FileNotFoundError("No se encontro el insumo: "+_nm)
POLI_V1 = gpd.read_file(_p_v1).to_crs(CRS_WEB)
POLI_OF = gpd.read_file(_p_of).to_crs(CRS_WEB)
# Bounds del poligono oficial (es el mas amplio) para encuadrar.
_b=POLI_OF.total_bounds
BO_BOUNDS=[[float(_b[1]),float(_b[0])],[float(_b[3]),float(_b[2])]]
# Carpeta de salida de mapas.
OUT_DIR=None
for _d in [os.path.join("..","data_processed_geo"),"data_processed_geo","/content/data_processed_geo"]:
    try:
        os.makedirs(_d,exist_ok=True); OUT_DIR=os.path.abspath(_d); break
    except Exception: continue
MAPAS_DIR=os.path.abspath(os.path.join(os.path.dirname(OUT_DIR.rstrip(os.sep)),"mapa_html","Barrio_obrero"))
os.makedirs(MAPAS_DIR,exist_ok=True)
print("V1 n_features:",len(POLI_V1),"| oficial n_features:",len(POLI_OF))
print("Bounds oficial:",BO_BOUNDS)
print("Salida de mapas HTML:",MAPAS_DIR)

## 3. Carga del IPM por manzana (45 manzanas) y areas

In [ ]:
# Capa de manzanas con IPM y las 15 privaciones (45 manzanas del limite oficial).
ipm_mz = gpd.read_file(_p_ipm)
ipm_mz = ipm_mz.set_crs(epsg=CRS_WEB) if ipm_mz.crs is None else ipm_mz.to_crs(CRS_WEB)
ipm_mz = ipm_mz[ipm_mz.geometry.notna()].reset_index(drop=True)
ipm_mz["ipm"] = pd.to_numeric(ipm_mz["ipm"], errors="coerce").fillna(0.0)
# Variables de privacion (nombres DANE). Normalizo a minuscula para el front.
VARS=["analf_","bajo_","infancia_","inasis_","rezago_","trab_infan_","depen_","infor_",
      "salud_","asegu_","haci_","pared_","excre_","pisos_","agua_"]
COLMAP={"trab_infan_":"TRAB_INFAN"}  # esta columna no lleva guion final en el geojson
for v in VARS:
    src = COLMAP.get(v, v.upper())
    ipm_mz[v] = pd.to_numeric(ipm_mz[src], errors="coerce").fillna(0.0) if src in ipm_mz.columns else 0.0
# Area de cada manzana en m2 (CRS metrico EPSG:3116 - MAGNA Bogota).
ipm_mz["area_m2"] = ipm_mz.to_crs(3116).geometry.area.values
# Marco de manzana sin poblacion: IPM == 0 en todas las privaciones clave.
ipm_mz["sin_pob"] = (ipm_mz["ipm"]==0) & (ipm_mz["bajo_"]==0) & (ipm_mz["infor_"]==0)
print("Manzanas IPM:",len(ipm_mz),"| sin poblacion:",int(ipm_mz['sin_pob'].sum()))
print("IPM (todas) media: %.2f | IPM (con poblacion) media: %.2f" % (
    ipm_mz['ipm'].mean(), ipm_mz.loc[~ipm_mz['sin_pob'],'ipm'].mean()))

## 4. Agregados IPM por poligono (V1 y oficial)

In [ ]:
VAR_LABEL={"analf_":"Analfabetismo","bajo_":"Bajo logro educativo","infancia_":"Barreras primera infancia",
  "inasis_":"Inasistencia escolar","rezago_":"Rezago escolar","trab_infan_":"Trabajo infantil",
  "depen_":"Dependencia economica","infor_":"Trabajo informal","salud_":"Barreras de salud",
  "asegu_":"Sin aseguramiento","haci_":"Hacinamiento critico","pared_":"Paredes precarias",
  "excre_":"Sin alcantarillado","pisos_":"Pisos precarios","agua_":"Sin acceso a agua"}
# Dimensiones DANE oficiales (agrupacion de las 15 variables).
DIMENSIONES={
  "Educacion":["analf_","bajo_","infancia_","inasis_","rezago_"],
  "Ninez y juventud":["infancia_","inasis_","rezago_","trab_infan_"],
  "Trabajo":["trab_infan_","depen_","infor_"],
  "Salud":["salud_","asegu_"],
  "Vivienda y servicios":["haci_","pared_","pisos_","agua_","excre_"],
}
def _agregar(sub, nombre, area_m2):
    """Resumen IPM de un subconjunto de manzanas (excluye sin poblacion del IPM promedio)."""
    con_pob = sub[~sub["sin_pob"]] if "sin_pob" in sub.columns else sub
    ipm_prom = float(con_pob["ipm"].mean()) if len(con_pob) else 0.0
    priv = {v: round(float(sub[v].mean()),1) if len(sub) else 0.0 for v in VARS}
    dim = {d: round(float(np.mean([priv[x] for x in xs])),1) for d,xs in DIMENSIONES.items()}
    return {"nombre":nombre, "n_manzanas":int(len(sub)), "n_con_pob":int(len(con_pob)),
            "area_m2":round(float(area_m2),1), "area_ha":round(float(area_m2)/10000.0,2),
            "ipm_prom":round(ipm_prom,1), "ipm_max":round(float(sub['ipm'].max()) if len(sub) else 0.0,1),
            "priv":priv, "dim":dim}
# Manzanas dentro de cada poligono por centroide (usar CRS metrico para centroides correctos).
cent = ipm_mz.to_crs(3116).geometry.centroid
cent_wgs = gpd.GeoSeries(cent, crs=3116).to_crs(CRS_WEB)
ipm_mz["cx"]=cent_wgs.x.values; ipm_mz["cy"]=cent_wgs.y.values
_v1u = POLI_V1.to_crs(3116).geometry.union_all()
_ofu = POLI_OF.to_crs(3116).geometry.union_all()
mask_v1 = cent.within(_v1u).values
mask_of = cent.within(_ofu).values
sub_v1 = ipm_mz[mask_v1].copy(); sub_of = ipm_mz[mask_of].copy()
AG_V1 = _agregar(sub_v1, "Poligono V1", POLI_V1.to_crs(3116).geometry.area.sum())
AG_OF = _agregar(sub_of, "Poligono oficial (45 mz)", POLI_OF.to_crs(3116).geometry.area.sum())
print("V1 ->", AG_V1["n_manzanas"], "mz | IPM", AG_V1["ipm_prom"], "| area ha", AG_V1["area_ha"])
print("Oficial ->", AG_OF["n_manzanas"], "mz | IPM", AG_OF["ipm_prom"], "| area ha", AG_OF["area_ha"])

## 5. Exportacion de lamina (deck) - helper reutilizable

In [ ]:
# Helper reutilizable para inyectar en un mapa Folium UN boton de exportacion
# de lamina con estructura de deck (PMU). Captura el contenedor del mapa como
# imagen y la compone dentro de una lamina HTML (encabezado + 2 columnas +
# pie) que se exporta a PDF (una pagina). Este archivo es solo una plantilla
# de trabajo; su contenido se inserta como celda en los notebooks.
#
# Decision CORS/tiles: html2canvas no puede rasterizar los tiles base de
# Google/Esri por restricciones CORS (saldrian en blanco o rotos). Por eso,
# antes de capturar el mapa se oculta el panel de tiles base (.leaflet-tile-pane)
# y se pinta un fondo claro (#F4F6F9) en el contenedor; se capturan solo los
# overlays vectoriales (contorno, manzanas, puntos, calor). La imagen del mapa
# va en la columna derecha de la lamina.
def agregar_export(mapa, titulo, slug):
    """Inyecta en 'mapa' (folium.Map) un control flotante con UN boton:
    'Exportar lamina (deck)'. Al pulsar, captura el contenedor del mapa a una
    imagen y construye una lamina HTML con estructura de deck del PMU
    (encabezado dorado, 2 columnas: resumen/cifras clonadas del panel +
    mapa, pie con corte y fuente); captura la lamina y la exporta a PDF de
    una pagina (landscape) con dialogo "Guardar como" (showSaveFilePicker) y
    fallback a descarga. Nombre: deck_<slug>_<AAAA-MM-DD>.pdf. Debe llamarse
    ANTES de mapa.save(...).
    - titulo: titulo visible en el encabezado de la lamina.
    - slug: nombre corto para el archivo.
    """
    map_id = mapa.get_name()  # id del div .folium-map de ESTE mapa
    # Librerias por CDN (sin build).
    cdn = (
        '<script src="https://cdn.jsdelivr.net/npm/html2canvas@1.4.1/dist/html2canvas.min.js"></script>'
        '<script src="https://cdn.jsdelivr.net/npm/jspdf@2.5.1/dist/jspdf.umd.min.js"></script>'
    )
    mapa.get_root().header.add_child(folium.Element(cdn))
    # Control + logica. Se escapa como string de Python; en JS se usa map_id,
    # titulo y slug concretos de ESTE mapa (asi funciona con varios mapas).
    titulo_js = titulo.replace("\\", "\\\\").replace("'", "\\'")
    slug_js = slug.replace("\\", "\\\\").replace("'", "\\'")
    bloque = """
<div id="exportctl___MID__" class="export-ctl" style="position:fixed;top:150px;left:10px;z-index:1002;
     display:flex;flex-direction:column;gap:6px;font-family:system-ui,Segoe UI,sans-serif">
  <button id="btndeck___MID__" style="cursor:pointer;border:1px solid #333;background:#fff;
     color:#222;padding:6px 10px;border-radius:6px;box-shadow:0 2px 6px rgba(0,0,0,.25);
     font-size:12px;font-weight:600">Exportar lamina (deck)</button>
</div>
<script>
(function(){
  var MAP_ID = '__MID__';
  var TITULO = '__TITULO__';
  var SLUG = '__SLUG__';
  var FUENTE = 'ITT Ciudad Paraiso - CDO Alcaldia de Santiago de Cali';
  var DORADO = '#8A6D1A';
  function fechaISO(){
    var d = new Date();
    var mm = ('0'+(d.getMonth()+1)).slice(-2);
    var dd = ('0'+d.getDate()).slice(-2);
    return d.getFullYear()+'-'+mm+'-'+dd;
  }
  function getContainer(){
    return document.getElementById(MAP_ID);
  }
  // Paso (a): captura SOLO el contenedor del mapa a un canvas, ocultando antes
  // los botones de export, el selector de capas, los tiles base (CORS), el
  // panel flotante de graficas y el slider de intensidad del calor; restaura
  // todo despues. Devuelve el canvas via callback.
  function capturarMapa(cont, cb){
    var ctl = document.getElementById('exportctl_'+MAP_ID);
    if(ctl) ctl.style.display = 'none';
    var layerCtls = cont.querySelectorAll('.leaflet-control-layers');
    layerCtls.forEach(function(lc){ lc.dataset.prevDisplay = lc.style.display; lc.style.display = 'none'; });
    // Panel de graficas flotante (#seg-panel o #eu-panel) y slider del calor.
    var panel = cont.querySelector('#seg-panel, #eu-panel, #ipm-panel, #cmp-panel');
    var prevPanelDisplay = null;
    if(panel){ prevPanelDisplay = panel.style.display; panel.style.display = 'none'; }
    var slider = cont.querySelector('#heat-intensity-ctrl');
    var prevSliderDisplay = null;
    if(slider){ prevSliderDisplay = slider.style.display; slider.style.display = 'none'; }
    // CORS: ocultar tiles base y aplicar fondo claro.
    var tilePanes = cont.querySelectorAll('.leaflet-tile-pane');
    var fondoPrev = cont.style.background;
    cont.style.background = '#F4F6F9';
    tilePanes.forEach(function(tp){ tp.dataset.prevOpacity = tp.style.opacity; tp.style.opacity = '0'; });
    function restaurar(){
      if(ctl) ctl.style.display = '';
      layerCtls.forEach(function(lc){ lc.style.display = ''; });
      if(panel) panel.style.display = (prevPanelDisplay || '');
      if(slider) slider.style.display = (prevSliderDisplay || '');
      tilePanes.forEach(function(tp){ tp.style.opacity = tp.dataset.prevOpacity || ''; });
      cont.style.background = fondoPrev;
    }
    html2canvas(cont, {useCORS:true, backgroundColor:'#F4F6F9', logging:false, scale:2})
      .then(function(canvas){ restaurar(); cb(null, canvas); })
      .catch(function(err){ restaurar(); cb(err, null); });
  }
  // Paso previo: asegura el panel #seg-panel/#eu-panel expandido (si usa
  // <details>) para que Chart.js tenga el canvas visible y pintado, luego
  // captura el canvas real del grafico a un dataURL PNG. Devuelve null si
  // no hay canvas o esta vacio. cb(dataUrl|null).
  function capturarGrafico(cb){
    var panel = document.querySelector('#seg-panel, #eu-panel, #ipm-panel, #cmp-panel');
    if(!panel){ cb(null); return; }
    var dets = panel.querySelectorAll('details');
    dets.forEach(function(d){ d.setAttribute('open', 'open'); });
    // Pequeno tiempo para que el canvas quede visible y Chart.js pinte.
    setTimeout(function(){
      var cv = panel.querySelector('canvas[id*="chart"]') || panel.querySelector('canvas');
      if(!cv || !cv.width || !cv.height){ cb(null); return; }
      try { cb(cv.toDataURL('image/png')); }
      catch(e){ cb(null); }
    }, 250);
  }
  // Paso (b): construye el div-lamina temporal con estructura de deck.
  // La imagen del mapa (canvas de (a)) va en la columna derecha; el panel
  // existente (si lo hay) se clona en la columna izquierda.
  function construirLamina(mapaDataUrl, graficoDataUrl){
    var lam = document.createElement('div');
    lam.id = 'deck-lamina';
    lam.style.cssText = 'position:fixed;top:0;left:0;z-index:2147483647;'+
      'width:1200px;background:#FFFFFF;color:#333;'+
      'font-family:system-ui,Segoe UI,sans-serif;'+
      'box-sizing:border-box;overflow:hidden;';
    // ENCABEZADO (barra con titulo + linea de seccion gris + borde dorado).
    var head = document.createElement('div');
    head.style.cssText = 'padding:16px 22px 12px 22px;border-bottom:3px solid '+DORADO+';';
    head.innerHTML =
      '<div style="font-size:20px;font-weight:700;color:#222;line-height:1.2">'+TITULO+'</div>'+
      '<div style="font-size:12px;color:#888;margin-top:4px">ITT Ciudad Paraiso - Lamina de deck (PMU)</div>';
    lam.appendChild(head);
    // CUERPO en 2 columnas (grid).
    var body = document.createElement('div');
    body.style.cssText = 'display:grid;grid-template-columns:35% 65%;gap:16px;'+
      'padding:16px 22px;align-items:start;';
    // Columna izquierda: resumen / cifras (clon del panel si existe).
    var colIzq = document.createElement('div');
    colIzq.style.cssText = 'min-width:0;';
    var panelOrig = document.querySelector('#seg-panel, #eu-panel, #ipm-panel, #cmp-panel');
    if(panelOrig){
      var clon = panelOrig.cloneNode(true);
      clon.removeAttribute('id');
      // Sacar el clon del flujo fijo para que fluya dentro de la columna.
      clon.style.position = 'static';
      clon.style.top = 'auto'; clon.style.left = 'auto';
      clon.style.right = 'auto'; clon.style.bottom = 'auto';
      clon.style.zIndex = 'auto';
      clon.style.maxWidth = 'none';
      clon.style.width = '100%';
      clon.style.boxShadow = 'none';
      // Expandir cualquier <details> para que la serie se vea.
      var dets = clon.querySelectorAll('details');
      dets.forEach(function(d){ d.setAttribute('open', 'open'); });
      // El canvas de Chart.js se clona VACIO (los canvas no conservan su
      // contenido al clonar). Lo sustituimos por un <img> con la imagen
      // capturada del canvas real para que el grafico SE VEA en la lamina.
      if(graficoDataUrl){
        var cvClon = clon.querySelector('canvas');
        if(cvClon){
          var imgChart = document.createElement('img');
          imgChart.src = graficoDataUrl;
          imgChart.style.cssText = 'display:block;width:100%;height:auto;';
          if(cvClon.parentNode){ cvClon.parentNode.replaceChild(imgChart, cvClon); }
        }
      }
      colIzq.appendChild(clon);
    } else {
      colIzq.innerHTML =
        '<div style="font-size:15px;font-weight:700;color:#222;margin-bottom:6px">'+TITULO+'</div>'+
        '<div style="font-size:13px;color:#666">Resumen no disponible</div>';
    }
    // Columna derecha: imagen del mapa escalada al ancho de la columna.
    var colDer = document.createElement('div');
    colDer.style.cssText = 'min-width:0;';
    var img = document.createElement('img');
    img.src = mapaDataUrl;
    img.style.cssText = 'display:block;width:100%;height:auto;'+
      'border:1px solid #d9dde3;border-radius:6px;';
    colDer.appendChild(img);
    body.appendChild(colIzq);
    body.appendChild(colDer);
    lam.appendChild(body);
    // PIE (corte + fuente + nota).
    var pie = document.createElement('div');
    pie.style.cssText = 'padding:10px 22px 14px 22px;border-top:1px solid #e2e5ea;'+
      'font-size:11px;color:#777;';
    pie.innerHTML = 'Corte: '+new Date().toLocaleString()+
      '  |  Fuente: '+FUENTE+'  |  Mapa ilustrativo';
    lam.appendChild(pie);
    document.body.appendChild(lam);
    return lam;
  }
  // Dialogo "Guardar como" (Chrome/Edge) con fallback a descarga clasica.
  function guardarPDF(blob, nombre){
    function descargaFallback(){
      var url = URL.createObjectURL(blob);
      var a = document.createElement('a');
      a.href = url; a.download = nombre;
      document.body.appendChild(a); a.click(); document.body.removeChild(a);
      setTimeout(function(){ URL.revokeObjectURL(url); }, 1500);
    }
    if(window.showSaveFilePicker){
      var tipos = [{description:'Documento PDF', accept:{'application/pdf':['.pdf']}}];
      window.showSaveFilePicker({suggestedName:nombre, types:tipos})
        .then(function(handle){ return handle.createWritable(); })
        .then(function(writable){ return writable.write(blob).then(function(){ return writable.close(); }); })
        .catch(function(e){
          // Si el usuario cancela (AbortError) no se descarga nada; otros -> fallback.
          if(e && e.name === 'AbortError'){ return; }
          descargaFallback();
        });
    } else {
      descargaFallback();
    }
  }
  // Bounds del poligono de la zona de estudio (Ciudad Paraiso) para
  // reencuadrar el mapa SOLO en la generacion de la lamina (no afecta la
  // vista interactiva previa del usuario; tras exportar queda en este
  // encuadre pero la interactividad se mantiene).
  var ZONA_BOUNDS = L.latLngBounds([[3.442445,-76.53259],[3.44791,-76.526004]]);
  function exportar(){
    var cont = getContainer();
    if(!cont){ alert('No se encontro el contenedor del mapa.'); return; }
    // 1) Forzar el panel a 'open' y capturar el grafico real a dataURL.
    capturarGrafico(function(graficoDataUrl){
      var mp = window[MAP_ID];
      // 2) fitBounds al poligono. maxZoom ENTERO evita el zoom fraccional que
      //    descoloca las capas en html2canvas (desfase al imprimir).
      var capturado = false;
      function trasEncuadre(){
        if(capturado){ return; }
        capturado = true;
        try { if(mp){ mp.invalidateSize(false); } } catch(e){}
        // 3) Margen corto tras asentar panes/tiles, luego capturar.
        setTimeout(function(){
        // 4) Capturar el mapa (ocultando botones/tiles/panel/slider).
        capturarMapa(cont, function(err, canvasMapa){
          if(err){ alert('Error al capturar el mapa: '+err); return; }
          var mapaDataUrl = canvasMapa.toDataURL('image/jpeg', 0.95);
          // 5) Construir lamina (cifras + grafico como imagen + mapa).
          var lam = construirLamina(mapaDataUrl, graficoDataUrl);
          function limpiar(){
            if(lam && lam.parentNode) lam.parentNode.removeChild(lam);
          }
          // 6) Capturar la lamina y generar el PDF.
          html2canvas(lam, {backgroundColor:'#FFFFFF', logging:false, scale:2, useCORS:true})
            .then(function(canvas){
              limpiar();
              var imgData = canvas.toDataURL('image/jpeg', 0.95);
              var jsPDFns = (window.jspdf && window.jspdf.jsPDF) ? window.jspdf.jsPDF : window.jsPDF;
              var w = canvas.width, hh = canvas.height;
              var orient = (w >= hh) ? 'landscape' : 'portrait';
              var doc = new jsPDFns({orientation:orient, unit:'pt', format:[w, hh]});
              doc.addImage(imgData, 'JPEG', 0, 0, w, hh);
              var blob = doc.output('blob');
              var nombre = 'deck_'+SLUG+'_'+fechaISO()+'.pdf';
              guardarPDF(blob, nombre);
            })
            .catch(function(e){
              limpiar();
              alert('Error al exportar la lamina: '+e);
            });
        });
        }, 350);
      }
      // Disparar fitBounds y esperar 'moveend'; fallback por timeout si no llega.
      try {
        if(mp){
          mp.once('moveend', trasEncuadre);
          mp.fitBounds(ZONA_BOUNDS, {padding:[10,10], maxZoom: 17, animate: false});
          setTimeout(trasEncuadre, 1200);
        } else {
          trasEncuadre();
        }
      } catch(e){ trasEncuadre(); }
    });
  }
  function init(){
    var b = document.getElementById('btndeck_'+MAP_ID);
    if(b) b.addEventListener('click', function(){ exportar(); });
  }
  if(document.readyState === 'loading'){
    document.addEventListener('DOMContentLoaded', init);
  } else { init(); }
})();
</script>
"""
    bloque = (bloque.replace("__MID__", map_id)
                    .replace("__TITULO__", titulo_js)
                    .replace("__SLUG__", slug_js))
    mapa.get_root().html.add_child(folium.Element(bloque))
    return mapa

## 6. Construccion del mapa (Folium): 3 poligonos + seleccion dinamica

In [ ]:
# Mapa 5: comparacion trilateral. Mapa base (3 tiles) + mascara + las 3 capas:
#   - Poligono V1 (azul, linea discontinua).
#   - Poligono oficial 45 mz (verde, linea continua).
#   - Coropleta IPM por manzana (contexto, escala 4 niveles daltonica).
#   - Capa de seleccion dinamica (Leaflet.draw) que captura manzanas con turf.js.
import folium
from folium.plugins import Draw

# Escala IPM 4 niveles (cortes del proyecto 8/15/25), colores daltonicos azul->naranja.
IPM_NIVELES = [
    (0.0, 8.0,   "Muy bajo / Bajo",       "#2c7fb8"),
    (8.1, 15.0,  "Medio / Tipico urbano", "#a6bddb"),
    (15.1, 25.0, "Alto",                  "#fdae61"),
    (25.1, 100.0,"Critico / Severo",      "#e34a33"),
]
def _niv_idx(v):
    v=float(v)
    for i,(lo,hi,_n,_c) in enumerate(IPM_NIVELES):
        if v<=hi: return i
    return len(IPM_NIVELES)-1
def _cat_nombre(v): return IPM_NIVELES[_niv_idx(v)][2]
def _color_ipm(v): return IPM_NIVELES[_niv_idx(v)][3]

_c = POLI_OF.geometry.union_all().centroid
m = folium.Map(location=[_c.y,_c.x], zoom_start=15, tiles=None, control_scale=True,
               zoom_snap=0.25, zoom_delta=0.1, max_zoom=22)
m.get_root().width="100%"; m.get_root().height="100%"
m.get_root().header.add_child(folium.Element(
    "<style>html,body{height:100%;margin:0;padding:0}"
    ".folium-map{position:absolute;top:0;bottom:0;left:0;right:0;width:100%;height:100%}"
    "/* Jerarquia de apilamiento (desacople de controles). */"
    ".leaflet-top,.leaflet-bottom{z-index:1000}"
    "#cmp-panel{z-index:1001;background:rgba(255,255,255,.95);backdrop-filter:blur(4px);"
    "-webkit-backdrop-filter:blur(4px);box-shadow:0 4px 15px rgba(0,0,0,.2)}"
    ".export-ctl{z-index:1002}"
    ".export-dropdown{z-index:1002}</style>"))
folium.TileLayer(tiles="https://mt1.google.com/vt/lyrs=m&x={x}&y={y}&z={z}",
    attr="Google", name="Google Streets", show=True, max_zoom=22, max_native_zoom=20).add_to(m)
folium.TileLayer(tiles="https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}",
    attr="Esri", name="Esri Satelite", show=False, max_zoom=22, max_native_zoom=20).add_to(m)
folium.TileLayer(tiles="https://mt1.google.com/vt/lyrs=s&x={x}&y={y}&z={z}",
    attr="Google", name="Google Satelite", show=False, max_zoom=22, max_native_zoom=20).add_to(m)

# Mascara oscura fuera del poligono oficial (foco en la zona).
_zg = POLI_OF.geometry.union_all()
_mundo=[[-180,-85],[180,-85],[180,85],[-180,85],[-180,-85]]; _ag=[]
for _gm in (list(_zg.geoms) if _zg.geom_type=="MultiPolygon" else [_zg]):
    _ag.append([[x,y] for x,y in _gm.exterior.coords])
folium.GeoJson({"type":"Feature","properties":{},"geometry":{"type":"Polygon","coordinates":[_mundo]+_ag}},
    name="Mascara (foco en la zona)", control=False,
    style_function=lambda f:{"fillColor":"#000","color":"#000","weight":0,"fillOpacity":0.18}).add_to(m)

# Coropleta IPM por manzana (contexto; apagada al inicio para no tapar los contornos).
_fg_ipm=folium.FeatureGroup(name="IPM: indice por manzana (45)", show=False)
for _,r in ipm_mz.iterrows():
    _v=float(r["ipm"]); _sp=bool(r["sin_pob"])
    _col = "#bdbdbd" if _sp else _color_ipm(_v)
    _tt = "Sin poblacion" if _sp else ("IPM: %.1f%% (%s)" % (_v,_cat_nombre(_v)))
    folium.GeoJson(r["geometry"].__geo_interface__,
        style_function=(lambda f,c=_col:{"fillColor":c,"color":"#ffffff","weight":0.5,"fillOpacity":0.7}),
        tooltip=_tt).add_to(_fg_ipm)
_fg_ipm.add_to(m)

# Poligono oficial (verde, linea continua).
folium.GeoJson(POLI_OF.to_json(), name="Poligono oficial (45 manzanas)", show=True,
    style_function=lambda f:{"color":OKI_VERDE,"weight":4,"fill":False}).add_to(m)
# Poligono V1 (azul, linea discontinua).
folium.GeoJson(POLI_V1.to_json(), name="Poligono V1 (16 manzanas)", show=True,
    style_function=lambda f:{"color":OKI_AZUL,"weight":4,"dashArray":"10 6","fill":False}).add_to(m)

# Herramienta de dibujo (Leaflet.draw): circulo y poligono libre para el 3er poligono.
Draw(export=False, position="topleft",
     draw_options={"polyline":False,"rectangle":False,"marker":False,"circlemarker":False,
                   "circle":{"shapeOptions":{"color":OKI_NARANJA,"weight":3,"fillColor":OKI_NARANJA,"fillOpacity":0.12}},
                   "polygon":{"shapeOptions":{"color":OKI_NARANJA,"weight":3,"fillColor":OKI_NARANJA,"fillOpacity":0.12}}},
     edit_options={"edit":False,"remove":True}).add_to(m)

m.fit_bounds(BO_BOUNDS, padding=(30,30))
folium.LayerControl(collapsed=True, position="topright").add_to(m)
print("Mapa base + 3 capas + herramienta de dibujo construidos.")

## 7. Panel comparativo trilateral + logica de seleccion dinamica (turf.js)

In [ ]:
# Panel flotante #cmp-panel con la ficha comparativa (V1 vs oficial vs aledano),
# grafico de barras agrupadas (Chart.js, 3 series Okabe-Ito) y bloque ITT rotulado
# como no disponible. La seleccion dinamica (3er poligono) se resuelve en el
# navegador con turf.js: al dibujar un circulo/poligono se capturan las manzanas
# cuyo centroide quede contenido, se resaltan y se recalcula el promedio IPM y las
# dimensiones; se actualizan la ficha y la 3a serie del grafico.
MID = m.get_name()

# GeoJSON de manzanas con atributos IPM para turf.js (centroide precalculado).
_feats=[]
for _,r in ipm_mz.iterrows():
    _props={"ipm":float(r["ipm"]), "sin_pob":bool(r["sin_pob"]),
            "area_m2":float(r["area_m2"]), "cx":float(r["cx"]), "cy":float(r["cy"])}
    for v in VARS: _props[v]=float(r[v])
    _feats.append({"type":"Feature","properties":_props,
                   "geometry":r["geometry"].__geo_interface__})
MZ_GJ={"type":"FeatureCollection","features":_feats}

# Config que viaja al front: etiquetas, dimensiones, agregados V1/oficial, colores.
CFG={
  "vars":VARS, "varLabel":VAR_LABEL, "dims":DIMENSIONES,
  "dimOrden":list(DIMENSIONES.keys()),
  "v1":AG_V1, "of":AG_OF,
  "colV1":OKI_AZUL, "colOf":OKI_VERDE, "colSel":OKI_NARANJA, "gris":OKI_GRIS, "bg":BG,
  "niveles":[[lo,hi,nom,col] for (lo,hi,nom,col) in IPM_NIVELES],
}
_cfg_js=json.dumps(CFG, ensure_ascii=False)
_mz_js=json.dumps(MZ_GJ, ensure_ascii=False)

_panel = r"""
<link rel="stylesheet" href="https://cdn.jsdelivr.net/npm/leaflet-draw@1.0.4/dist/leaflet.draw.css"/>
<script src="https://cdn.jsdelivr.net/npm/@turf/turf@6/turf.min.js"></script>
<script src="https://cdn.jsdelivr.net/npm/chart.js"></script>
<div id="cmp-panel" style="position:fixed;bottom:25px;right:20px;z-index:1001;
     background:__BG__;max-width:380px;width:min(380px,44vw);max-height:80vh;overflow-y:auto;
     padding:11px 13px;border-radius:10px;box-shadow:0 4px 15px rgba(0,0,0,.2);
     font-family:system-ui,Segoe UI,sans-serif;font-size:12px;color:__GRIS__">
  <div style="display:flex;align-items:center;justify-content:space-between;gap:8px">
    <div style="font-weight:700;font-size:13px">Comparacion territorial - Barrio Obrero</div>
    <button id="cmp-toggle" title="Minimizar / expandir" style="cursor:pointer;border:1px solid #bbb;background:#fff;color:#333;width:24px;height:22px;border-radius:5px;font-size:13px;line-height:1;font-weight:700;flex:0 0 auto">_</button>
  </div>
  <div id="cmp-body">
  <div style="font-size:10px;color:#666;margin:2px 0 8px 0">IPM por manzana - % de personas (vacios = 0). Fuente: Censo 2018 (DANE, CNPV).</div>
  <div style="display:flex;gap:6px;margin-bottom:8px;font-size:10px">
    <span style="display:flex;align-items:center;gap:4px"><i style="width:14px;height:3px;background:__CV1__;display:inline-block"></i>V1</span>
    <span style="display:flex;align-items:center;gap:4px"><i style="width:14px;height:0;border-top:3px dashed __CV1__;display:inline-block"></i></span>
    <span style="display:flex;align-items:center;gap:4px"><i style="width:14px;height:3px;background:__COF__;display:inline-block"></i>Oficial (45 mz)</span>
    <span style="display:flex;align-items:center;gap:4px"><i style="width:12px;height:12px;background:__CSEL__;opacity:.5;border:1px solid __CSEL__;display:inline-block;border-radius:50%"></i>Aledano</span>
  </div>
  <table style="width:100%;border-collapse:collapse;font-size:11px;margin-bottom:8px">
    <thead><tr style="border-bottom:1px solid #ccc">
      <th style="text-align:left;padding:3px 2px">Metrica</th>
      <th style="text-align:right;padding:3px 2px;color:__CV1__">V1</th>
      <th style="text-align:right;padding:3px 2px;color:__COF__">Oficial</th>
      <th style="text-align:right;padding:3px 2px;color:__CSEL__">Aledano</th>
    </tr></thead>
    <tbody id="cmp-tbody"></tbody>
  </table>
  <div id="cmp-sel-aviso" style="font-size:10px;color:#888;margin:-2px 0 8px 0">
    Dibuja un circulo o poligono (herramienta arriba-izquierda) para calcular el area aledana.</div>
  <details open style="margin-bottom:6px">
    <summary style="cursor:pointer;font-weight:600;color:__CV1__;outline:none">Dimensiones DANE (promedio por poligono) / Ocultar</summary>
    <div style="position:relative;height:230px;width:100%;margin-top:6px"><canvas id="cmp-chart"></canvas></div>
    <div style="font-size:9px;color:#777;margin-top:3px">Escala 0-100 (% de personas). Dimensiones oficiales DANE.</div>
  </details>
  <details style="margin-bottom:2px">
    <summary style="cursor:pointer;font-weight:600;color:#999;outline:none">Dimensiones ITT (seguridad / movilidad) / Mostrar</summary>
    <div style="background:#f0e9e0;border:1px dashed #c8a96a;border-radius:6px;padding:8px;margin-top:6px;font-size:11px;color:#7a6a4a">
      <b>Sin datos ITT para Comuna 9 (pendiente de DATIC).</b><br>
      El dataset ITT real (hurtos, homicidios, rinas, siniestros) solo cubre Ciudad
      Paraiso (Comuna 3). Barrio Obrero (Comuna 9) aun no tiene eventos cargados a
      nivel de manzana, por lo que la comparacion se limita al IPM.
    </div>
  </details>
  </div>
</div>
<script>
(function(){
  var CFG=__CFG__;
  var MZ=__MZ__;
  var MID="__MID__";
  var serieSel=null;   // agregado de la seleccion dinamica (o null)
  var chart=null;
  var capaResaltada=null;

  function fmtNum(x){ return (typeof x==="number")? x.toLocaleString("es-CO") : "-"; }
  function fmtPct(x){ return (x==null)? "-" : (String(x).replace(".",",")+"%"); }

  // --- tabla comparativa ---
  function filasTabla(){
    var v1=CFG.v1, of=CFG.of, se=serieSel;
    function cel(a,b,c){ return "<td style='text-align:right;padding:3px 2px'>"+a+"</td>"+
      "<td style='text-align:right;padding:3px 2px'>"+b+"</td>"+
      "<td style='text-align:right;padding:3px 2px'>"+c+"</td>"; }
    var rows="";
    function fila(label,a,b,c){ return "<tr style='border-bottom:1px solid #eee'>"+
      "<td style='padding:3px 2px'>"+label+"</td>"+cel(a,b,c)+"</tr>"; }
    rows+=fila("Manzanas", fmtNum(v1.n_manzanas), fmtNum(of.n_manzanas), se?fmtNum(se.n_manzanas):"-");
    rows+=fila("Area (ha)", fmtNum(v1.area_ha), fmtNum(of.area_ha), se?fmtNum(se.area_ha):"-");
    rows+=fila("IPM promedio", fmtPct(v1.ipm_prom), fmtPct(of.ipm_prom), se?fmtPct(se.ipm_prom):"-");
    rows+=fila("IPM maximo", fmtPct(v1.ipm_max), fmtPct(of.ipm_max), se?fmtPct(se.ipm_max):"-");
    // Top 3 privaciones del oficial como referencia.
    var ord=CFG.vars.slice().sort(function(x,y){ return of.priv[y]-of.priv[x]; }).slice(0,3);
    ord.forEach(function(v){
      rows+=fila(CFG.varLabel[v],
        fmtPct(v1.priv[v]), fmtPct(of.priv[v]), se?fmtPct(se.priv[v]):"-");
    });
    document.getElementById("cmp-tbody").innerHTML=rows;
  }

  // --- grafico de barras agrupadas por dimension DANE ---
  function datosGrafico(){
    var dims=CFG.dimOrden;
    function serie(ag){ return dims.map(function(d){ return ag? (ag.dim[d]||0):0; }); }
    var ds=[
      {label:"V1", data:serie(CFG.v1), backgroundColor:CFG.colV1},
      {label:"Oficial", data:serie(CFG.of), backgroundColor:CFG.colOf},
    ];
    if(serieSel){ ds.push({label:"Aledano", data:serie(serieSel), backgroundColor:CFG.colSel}); }
    return {labels:dims, datasets:ds};
  }
  function dibujarGrafico(){
    var cv=document.getElementById("cmp-chart");
    if(!cv||typeof Chart==="undefined") return;
    var d=datosGrafico();
    if(chart){ chart.data=d; chart.update(); return; }
    chart=new Chart(cv.getContext("2d"),{type:"bar",data:d,
      options:{responsive:true,maintainAspectRatio:false,
        plugins:{legend:{position:"bottom",labels:{font:{size:10},boxWidth:12}}},
        scales:{y:{beginAtZero:true,max:100,ticks:{font:{size:9}},title:{display:true,text:"% de personas",font:{size:9}}},
                x:{ticks:{font:{size:8},maxRotation:30,minRotation:0}}}}});
  }

  // --- agregacion de la seleccion dinamica con turf.js ---
  function dentro(figura, feat){
    // feat trae centroide precalculado (cx, cy) en WGS84.
    var pt=turf.point([feat.properties.cx, feat.properties.cy]);
    try { return turf.booleanPointInPolygon(pt, figura); }
    catch(e){ return false; }
  }
  function circuloAPoligono(center, radiusMeters){
    // turf.circle devuelve un poligono aproximado para el point-in-polygon.
    return turf.circle([center.lng, center.lat], radiusMeters/1000.0, {steps:64, units:"kilometers"});
  }
  function agregar(seleccion){
    var n=seleccion.length;
    if(!n){ return {nombre:"Aledano", n_manzanas:0, n_con_pob:0, area_m2:0, area_ha:0,
                    ipm_prom:0, ipm_max:0, priv:{}, dim:{}}; }
    var areaM2=0, ipmSum=0, ipmCnt=0, ipmMax=0;
    var privSum={}; CFG.vars.forEach(function(v){ privSum[v]=0; });
    seleccion.forEach(function(f){
      var p=f.properties; areaM2+=p.area_m2||0;
      if(!p.sin_pob){ ipmSum+=p.ipm; ipmCnt++; }
      if(p.ipm>ipmMax) ipmMax=p.ipm;
      CFG.vars.forEach(function(v){ privSum[v]+=(p[v]||0); });
    });
    var priv={}; CFG.vars.forEach(function(v){ priv[v]=Math.round((privSum[v]/n)*10)/10; });
    var dim={};
    CFG.dimOrden.forEach(function(d){
      var xs=CFG.dims[d]; var s=0; xs.forEach(function(x){ s+=priv[x]; });
      dim[d]=Math.round((s/xs.length)*10)/10;
    });
    return {nombre:"Aledano", n_manzanas:n, n_con_pob:ipmCnt,
            area_m2:Math.round(areaM2*10)/10, area_ha:Math.round((areaM2/10000)*100)/100,
            ipm_prom: ipmCnt? Math.round((ipmSum/ipmCnt)*10)/10 : 0,
            ipm_max: Math.round(ipmMax*10)/10, priv:priv, dim:dim};
  }

  // --- resaltar manzanas capturadas en el mapa ---
  function resaltar(mapObj, seleccion){
    if(capaResaltada){ mapObj.removeLayer(capaResaltada); capaResaltada=null; }
    if(!seleccion.length) return;
    capaResaltada=L.geoJSON({type:"FeatureCollection",features:seleccion},
      {style:function(){ return {color:CFG.colSel,weight:1.2,fillColor:CFG.colSel,fillOpacity:0.45}; }});
    capaResaltada.addTo(mapObj);
  }

  function manejarFigura(mapObj, figura){
    var seleccion=MZ.features.filter(function(f){ return dentro(figura,f); });
    serieSel=agregar(seleccion);
    resaltar(mapObj, seleccion);
    var aviso=document.getElementById("cmp-sel-aviso");
    if(aviso){ aviso.innerHTML = seleccion.length
      ? ("Area aledana: <b>"+seleccion.length+"</b> manzanas capturadas. Vuelve a dibujar para reemplazar; usa la papelera para limpiar.")
      : "Ninguna manzana quedo dentro de la figura. Dibuja sobre las manzanas aledanas."; }
    filasTabla(); dibujarGrafico();
  }
  function limpiarSeleccion(){
    serieSel=null;
    if(capaResaltada){ try{ window[MID].removeLayer(capaResaltada);}catch(e){} capaResaltada=null; }
    var aviso=document.getElementById("cmp-sel-aviso");
    if(aviso){ aviso.innerHTML="Dibuja un circulo o poligono (herramienta arriba-izquierda) para calcular el area aledana."; }
    filasTabla(); dibujarGrafico();
  }

  function init(){
    filasTabla();
    setTimeout(dibujarGrafico, 300);
    var mapObj=window[MID];
    if(!mapObj||typeof L==="undefined"){ return; }
    mapObj.on(L.Draw.Event.CREATED, function(e){
      var layer=e.layer, figura;
      if(e.layerType==="circle"){ figura=circuloAPoligono(layer.getLatLng(), layer.getRadius()); }
      else { figura=layer.toGeoJSON(); }
      manejarFigura(mapObj, figura);
    });
    mapObj.on(L.Draw.Event.DELETED, function(){ limpiarSeleccion(); });
  }
  function initColapso(){
    var btn=document.getElementById("cmp-toggle");
    var body=document.getElementById("cmp-body");
    if(!btn||!body) return;
    btn.addEventListener("click", function(){
      var oculto=(body.style.display==="none");
      body.style.display = oculto ? "" : "none";
      btn.textContent = oculto ? "_" : "+";
      btn.title = oculto ? "Minimizar" : "Expandir";
    });
  }
  if(document.readyState==="complete"){ setTimeout(init,350); initColapso(); }
  else { window.addEventListener("load", function(){ setTimeout(init,350); initColapso(); }); }
})();
</script>
"""
_panel=(_panel.replace("__BG__",BG).replace("__GRIS__",OKI_GRIS)
             .replace("__CV1__",OKI_AZUL).replace("__COF__",OKI_VERDE).replace("__CSEL__",OKI_NARANJA)
             .replace("__CFG__",_cfg_js).replace("__MZ__",_mz_js).replace("__MID__",MID))
m.get_root().html.add_child(folium.Element(_panel))
print("Panel comparativo trilateral + logica turf.js inyectados.")

## 8. Exportacion del HTML

In [ ]:
agregar_export(m, "Comparacion territorial - Barrio Obrero", "bo_comparacion")
html=os.path.join(MAPAS_DIR,"mapa_comparacion_barrio_obrero.html")
m.save(html)
print("Mapa 5 (comparacion Barrio Obrero) exportado:", html)
m